## Machine Learning Assessment Project- Classification
###

Si Tang Lin

Student id: 476912

Data Science and Business Analytics

Machine learning 2

**Use python3.10 venv to run**


### 1. Import Libraries

In [1]:
## environment
import sys
assert sys.executable.endswith("/venv/bin/python"), sys.executable

In [ ]:
## Import data
import os
import math
import numpy as np
import pandas as pd

# ===============================
# Visualization
# ===============================
import matplotlib.pyplot as plt
import seaborn as sns

# ===============================
# Statistics
# ===============================
from scipy.stats import (
    chi2_contingency,
    pointbiserialr,
    ttest_ind,
    kruskal
)
from scipy.stats.mstats import winsorize

# ===============================
# Preprocessing & Encoding
# ===============================
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    PowerTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    LabelEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import category_encoders as ce

# ===============================
# Feature Engineering & Selection
# ===============================
from sklearn.feature_selection import (
    SelectKBest,
    f_classif,
    SelectFromModel,
    SequentialFeatureSelector
)
from sklearn.decomposition import PCA

# ===============================
# Model Selection
# ===============================
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV
)

# ===============================
# Models
# ===============================
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# ===============================
# Imbalanced Learning
# ===============================
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# ===============================
# Evaluation Metrics
# ===============================
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    precision_recall_fscore_support,
    make_scorer
)

# ===============================
# Misc
# ===============================
import sklearn


### 2. Load & Inspect Data

Date resource: https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success



In [3]:
## Load and inspect data
#os.chdir("/Users/ninalin/Desktop/dsba/winter semester 25/ml 2 /ml2 project data")

student_dropout = pd.read_csv("data/students dropout_data.csv")

print(student_dropout.head())
print(student_dropout.info())
print(student_dropout.describe())
print("Missing values:\n", student_dropout.isnull().sum())
# No missing value in our dataset
# All variables were cast to integers for processing.
# To ensure correct treatment of variable types (numeric, ordinal, nominal),
# we now restore each column to its appropriate original data type.


   Marital status  Application mode  Application order  Course  \
0               1                17                  5     171   
1               1                15                  1    9254   
2               1                 1                  5    9070   
3               1                17                  2    9773   
4               2                39                  1    8014   

   Daytime/evening attendance\t  Previous qualification  \
0                             1                       1   
1                             1                       1   
2                             1                       1   
3                             1                       1   
4                             0                       1   

   Previous qualification (grade)  Nacionality  Mother's qualification  \
0                           122.0            1                      19   
1                           160.0            1                       1   
2                         

### 3. Train and test split

In [4]:
# Train/ Test split

X_train, X_test, y_train, y_test = train_test_split(
    student_dropout.drop(columns="Target"),
    student_dropout["Target"],
    test_size=0.2,
    stratify=student_dropout["Target"],
    random_state=42
)

print("Successfully split train and test")


Successfully split train and test


### 3. Variable typing

In [5]:
## Assigning data type

X_train.columns = X_train.columns.str.strip()
nominal_cols = ["Marital status", "Application mode", "Course", "Nacionality",
                "Mother's occupation", "Father's occupation"]
ordinal_cols = ["Application order", "Previous qualification","Mother's qualification", "Father's qualification"]
numeric_cols = ["Previous qualification (grade)","Admission grade","Age at enrollment",
    'Curricular units 1st sem (evaluations)',
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 1st sem (credited)',
    'Curricular units 1st sem (enrolled)',
    'Curricular units 2nd sem (evaluations)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)',
    'Curricular units 2nd sem (credited)',
    'Curricular units 2nd sem (enrolled)','Unemployment rate',"Inflation rate","GDP"]
binary_cols = ["Displaced", "Daytime/evening attendance", "Educational special needs",
               "Debtor", "Tuition fees up to date", "Gender", "Scholarship holder", "International"]


In [6]:
## Restore to original data type

#X_train['Target'] = X_train['Target'].astype('category')

all_cols = nominal_cols + ordinal_cols + numeric_cols + binary_cols

for c in nominal_cols:
    if c in X_train.columns:
        X_train[c] = X_train[c].astype('category')

for c in ordinal_cols:
    if c in X_train.columns:
        X_train[c] = X_train[c].astype('category')

for c in binary_cols:
    if c in X_train.columns:
        X_train[c] = X_train[c].replace(r'^\s*$', np.nan, regex=True)
        X_train[c] = X_train[c].astype('category')

for c in numeric_cols:
    if c in X_train.columns:
        X_train[c] = pd.to_numeric(X_train[c], errors='coerce')

for c in all_cols:
    if c in X_train.columns:
        print(f"\n--- {c} ---")
        try:
            print(X_train[c].value_counts(dropna=False).head(10))
        except Exception as e:
            print("Error showing value_counts:", e)
print(X_train.info())



--- Marital status ---
Marital status
1    3144
2     296
4      73
5      19
6       4
3       3
Name: count, dtype: int64

--- Application mode ---
Application mode
1     1364
17     701
39     621
43     249
44     176
7      109
18      98
42      59
51      45
16      31
Name: count, dtype: int64

--- Course ---
Course
9500    607
9147    302
9238    291
9085    277
9773    267
9991    217
9670    213
9254    198
171     175
9070    174
Name: count, dtype: int64

--- Nacionality ---
Nacionality
1      3453
41       26
6        13
26       12
22       10
11        3
24        3
100       3
62        2
105       2
Name: count, dtype: int64

--- Mother's occupation ---
Mother's occupation
9     1261
4      661
5      429
3      269
2      252
7      221
0      115
1       83
6       77
90      54
Name: count, dtype: int64

--- Father's occupation ---
Father's occupation
9     819
7     501
5     410
4     310
3     304
8     260
10    210
6     198
2     167
1     113
Name: count, d

### 4. Near Zero Variance

In [7]:
## Define NZV function 
def near_zero_var(df, freq_cut=95/5, unique_cut=10):

    results = []
    for col in df.columns:
        counts = df[col].value_counts()

        if len(counts) > 1:
            freq_ratio = counts.iloc[0] / counts.iloc[1]
        else:
            freq_ratio = float('inf') 

        unique_ratio = len(counts) / len(df)

        high_freq_ratio = int(freq_ratio > freq_cut)
        low_unique_ratio = int(unique_ratio < unique_cut)

        results.append({
            'variable': col,
            'freq_ratio': freq_ratio,
            'unique_ratio': unique_ratio,
            'high_freq_ratio': high_freq_ratio,
            'low_unique_ratio': low_unique_ratio
        })
    results_df = pd.DataFrame(results)

    results_df = results_df.sort_values(by=['freq_ratio', 'unique_ratio'], ascending=[False, True])

    return results_df

In [8]:
## Apply to the dataset
X_train_nzv = near_zero_var(X_train, freq_cut=97/3, unique_cut=10)
X_train_nzv[(X_train_nzv['low_unique_ratio'] == 1) & (X_train_nzv['high_freq_ratio'] == 1)]

,variable,freq_ratio,unique_ratio,high_freq_ratio,low_unique_ratio
7,Nacionality,132.807692,0.005369,1,1
14,Educational special needs,85.317073,0.000565,1,1
20,International,40.151163,0.000565,1,1
21,Curricular units 1st sem (credited),39.896104,0.005934,1,1
27,Curricular units 2nd sem (credited),34.511111,0.005369,1,1


In [9]:
## Chi-square method to check if we can drop the variable

chi2_results = []

# Step 1: Near-zero variance variables
nzv_vars = [
    'Nacionality',
    'Educational special needs',
    'Curricular units 1st sem (credited)',
    'International',
    'Curricular units 2nd sem (credited)'
]

# Step 2: Multi-class Chi-square on NZV variables
for col in nzv_vars:
    
    if col not in X_train.columns:
        continue
    
    table = pd.crosstab(X_train[col], y_train)

    if table.shape[0] < 2:
        chi2_results.append([col, None, None])
        continue
    
    chi2, p, dof, expected = chi2_contingency(table)
    chi2_results.append([col, p, dof])

chi2_df = pd.DataFrame(chi2_results, columns=['variable', 'p_value', 'dof'])
print("\n===== Chi-square results on NZV variables (multi-class Target) =====")
print(chi2_df.sort_values('p_value'))


# Null Hypothesis: There is no statistically significant relationship between the two variables.
# There is a significant relationship between Curricular units 1st sem (credited), Curricular units 2nd sem (credited) and target variable, we proceed with further analysis on this variable.
# The relationship between Nacionality, International, Educational special needs and the target is not statistically significant.
### We drop "Nacionality", "International", "Educational special need"


===== Chi-square results on NZV variables (multi-class Target) =====
                              variable   p_value  dof
3                        International  0.237384    2
2  Curricular units 1st sem (credited)  0.255829   40
0                          Nacionality  0.331771   36
4  Curricular units 2nd sem (credited)  0.403942   36
1            Educational special needs  0.786371    2


In [10]:
## Initial Decisions on Near-Zero Variance

X_train.drop(['Nacionality'], axis=1, inplace=True)
X_train.drop(['International'], axis=1, inplace=True)
X_train.drop(['Educational special needs'], axis=1, inplace=True)

### 5. EDA& Feature Inspection

In [ ]:
# Correlation Matrix

numeric_cols = X_train.select_dtypes(include='number')

corr_matrix = numeric_cols.corr()
plt.figure(figsize=(16, 16))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True, linewidths=0.5)
plt.title("Correlation Matrix of Numeric Variables")
plt.show()

In [11]:
### Present an overall weak correlation through all variables

### Curricular units 1st sem/2nd sem might have collinearity, between variable"evaluations", "approved", "grade", "credited", "enrolled". We need to delete to prevent multicollineraity.
### Use PCA to reduce the dimension
### The plot imply the lower the grade the easier to dropout

In [12]:
## PCA- dimension reduction "Curricular units 1st sem" for"Curricular units 2nd sem"

def make_pca_feature(df, var_list, new_name):
   
    scaler = StandardScaler()
    scaled = scaler.fit_transform(df[var_list])
    pca = PCA(n_components=1)
    pc1 = pca.fit_transform(scaled)

    df[new_name] = pc1
    print(f"\n===== {new_name} =====")
    print("Explained Variance Ratio (PC1):", pca.explained_variance_ratio_[0])
    loadings = pd.Series(pca.components_[0], index=var_list)
    print("\nLoadings:")
    print(loadings)
    return df

# 1st sem
pca_vars_1st = [
    'Curricular units 1st sem (evaluations)',
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 1st sem (credited)',
    'Curricular units 1st sem (enrolled)'
]
# 2nd sem
pca_vars_2nd = [
    'Curricular units 2nd sem (evaluations)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)',
    'Curricular units 2nd sem (credited)',
    'Curricular units 2nd sem (enrolled)'
]

X_train = make_pca_feature(X_train, pca_vars_1st, "PCA_1st_sem")
X_train = make_pca_feature(X_train, pca_vars_2nd, "PCA_2nd_sem")


===== PCA_1st_sem =====
Explained Variance Ratio (PC1): 0.6516137042263388

Loadings:
Curricular units 1st sem (evaluations)    0.434001
Curricular units 1st sem (approved)       0.498263
Curricular units 1st sem (grade)          0.344700
Curricular units 1st sem (credited)       0.433358
Curricular units 1st sem (enrolled)       0.506715
dtype: float64

===== PCA_2nd_sem =====
Explained Variance Ratio (PC1): 0.6195386985093855

Loadings:
Curricular units 2nd sem (evaluations)    0.425441
Curricular units 2nd sem (approved)       0.505108
Curricular units 2nd sem (grade)          0.395655
Curricular units 2nd sem (credited)       0.399179
Curricular units 2nd sem (enrolled)       0.497976
dtype: float64


In [13]:
## Dropout ten variables
pca_vars_1st = [
    'Curricular units 1st sem (evaluations)',
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 1st sem (credited)',
    'Curricular units 1st sem (enrolled)'
]
pca_vars_2nd = [
    'Curricular units 2nd sem (evaluations)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)',
    'Curricular units 2nd sem (credited)',
    'Curricular units 2nd sem (enrolled)'
]
X_train = X_train.drop(columns=pca_vars_1st + pca_vars_2nd)
print(X_train.head())


     Marital status Application mode Application order Course  \
2283              1               43                 1   9238   
3874              1               39                 1   9130   
2281              1               44                 1   9003   
817               1               16                 1   9070   
404               1               17                 1   9500   

     Daytime/evening attendance Previous qualification  \
2283                          1                     39   
3874                          1                      1   
2281                          1                     39   
817                           1                      1   
404                           1                      1   

      Previous qualification (grade) Mother's qualification  \
2283                           120.0                      2   
3874                           133.1                      1   
2281                           140.0                      1   
817     

In [ ]:
# Correlation heatmap after PCA
numeric_df = X_train.select_dtypes(include=['int64', 'float64'])

plt.figure(figsize=(9, 6))
corr = numeric_df.corr()

sns.heatmap(
    corr,
    annot=False,
    cmap='coolwarm',
    linewidths=0.5,
    fmt='.2f',
    square=False
)
plt.title("Updated Correlation Matrix After PCA & Feature Removal", fontsize=14)
plt.show()

### 8. Feature Engineering

In [ ]:
# Check column
print(X_train.columns.tolist())


#### 8.1 Numerical variable

In [14]:
## Kruskal–Wallis test

numeric_cols = X_train.select_dtypes(include=['int64','float64']).columns.tolist()

for col in numeric_cols:
    groups = [X_train[col][y_train == level] for level in y_train.unique()]
    stat, p = kruskal(*groups)
    print(f'{col}: Kruskal-Wallis p-value = {p}')

## We can delete "inflation rate",as it might cause noise in the model

Previous qualification (grade): Kruskal-Wallis p-value = 1.3721510719206153e-11
Admission grade: Kruskal-Wallis p-value = 4.831504295730773e-14
Age at enrollment: Kruskal-Wallis p-value = 5.500052074114091e-69
Curricular units 1st sem (without evaluations): Kruskal-Wallis p-value = 4.3490083769862656e-07
Curricular units 2nd sem (without evaluations): Kruskal-Wallis p-value = 2.546295374120831e-07
Unemployment rate: Kruskal-Wallis p-value = 0.024197793538466544
Inflation rate: Kruskal-Wallis p-value = 0.25768241830761596
GDP: Kruskal-Wallis p-value = 0.00289743289596941
PCA_1st_sem: Kruskal-Wallis p-value = 8.956330156898277e-121
PCA_2nd_sem: Kruskal-Wallis p-value = 9.13464902526878e-177


In [15]:
## delete variables
X_train.drop(['Inflation rate'], axis=1, inplace=True)

In [ ]:
# Numeric variables x target variable plot

numeric_cols = X_train.select_dtypes(include=['int64','float64']).columns.tolist()

for col in numeric_cols:
    plt.figure(figsize=(3, 2))
    sns.boxplot(x=y_train, y=X_train[col])
    plt.title(f'{col} vs Claim Status')
    
    plt.show()

In [16]:
## Features add after plot inspect

epsilon = 1e-5
X_train["grade_ratio"] = (
    X_train["Admission grade"] /
    (X_train["Previous qualification (grade)"] + epsilon)
)

X_train["is_mature_student"] = np.where(
    X_train["Age at enrollment"] >= 25,
    1,
    0
)

In [18]:
#### DONT scaling
## For modeling purpose, we decide not to scale data

#numeric_features = [
    #'Previous qualification (grade)',
    #'Admission grade',
    #'Age at enrollment',
    #'Curricular units 1st sem (without evaluations)',
    #'Curricular units 2nd sem (without evaluations)',
    #'Unemployment rate',
    #'GDP',
    #'PCA_1st_sem',
    #'PCA_2nd_sem',
    #'grade_ratio',
    #'performance_drop']

#scaler = ColumnTransformer(
    #transformers=[
    #   ('scale', StandardScaler(), numeric_features)
    # ],
    #remainder='passthrough' )

#### 7.2 Nominal variable

In [ ]:
# Nominal variable x target variable distribition plot

nominal_cols = X_train.select_dtypes(include=['category','object']).columns.tolist()
palette = sns.color_palette("Pastel1", n_colors=y_train.nunique())

for col in nominal_cols:
    table = pd.crosstab(X_train[col], y_train)
    prop_df = table.div(table.sum(axis=1), axis=0) 
    ax = prop_df.plot(kind='bar', stacked=True, figsize=(6,3), color=palette)
    
    plt.title(f'Target Proportion by {col}', fontsize=12)
    plt.ylabel('Proportion')
    plt.xlabel(col)
    plt.xticks(rotation=45)
    plt.legend(title='Target', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [17]:
## Cramer's V
nominal_cols = [
    'Marital status', 
    'Application mode',
    'Application order',
    'Course',
    'Daytime/evening attendance',
    'Previous qualification',
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    'Displaced',
    'Debtor',
    'Tuition fees up to date',
    'Gender',
    'Scholarship holder'
]
def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return np.sqrt(chi2 / (n * (min(r - 1, k - 1))))


for col in nominal_cols:
    table = pd.crosstab(X_train[col], y_train)
    print(col, "Cramer's V =", cramers_v(table))


Marital status Cramer's V = 0.08806163900673357
Application mode Cramer's V = 0.24628207501854205
Application order Cramer's V = 0.10299227102002728
Course Cramer's V = 0.25450117512259623
Daytime/evening attendance Cramer's V = 0.08469456220080976
Previous qualification Cramer's V = 0.162432967115688
Mother's qualification Cramer's V = 0.15778888426413468
Father's qualification Cramer's V = 0.16859055672097406
Mother's occupation Cramer's V = 0.1904216496652308
Father's occupation Cramer's V = 0.18811027939977834
Displaced Cramer's V = 0.1173305308250013
Debtor Cramer's V = 0.25277226261611874
Tuition fees up to date Cramer's V = 0.43217621495458164
Gender Cramer's V = 0.2281194062420083
Scholarship holder Cramer's V = 0.3013575567082509


In [18]:
## Delete variables

X_train.drop(['Application order'], axis=1, inplace=True)
X_train.drop(['Marital status'], axis=1, inplace=True)
X_train.drop(['Daytime/evening attendance'], axis=1, inplace=True)

In [19]:
## Features add after plot inspect

X_train["performance_drop"] = (
    (X_train["Admission grade"] < X_train["Previous qualification (grade)"]) * 1
)
def age_group(age):
    if age < 22:
        return "traditional"
    elif age < 25:
        return "transition"
    else:
        return "mature"

X_train["age_group"] = X_train["Age at enrollment"].apply(age_group)

## Enoded

In [21]:
## Label Encoded for y variable

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
#y_test_encoded = le.transform(y_test)

In [22]:
## Encoding

high_card_cat = ['Course']

one_hot_cat = [
    'Application mode',
    'Previous qualification',
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    'Displaced',
    'Debtor',
    'Tuition fees up to date',
    'Gender',
    'Scholarship holder',
    'age_group'
]

num_cols = [
    'Previous qualification (grade)',
    'Admission grade',
    'Age at enrollment',
    'Curricular units 1st sem (without evaluations)',
    'Curricular units 2nd sem (without evaluations)',
    'Unemployment rate',
    'GDP',
    'PCA_1st_sem',
    'PCA_2nd_sem',
    'grade_ratio',
    'is_mature_student',
    'performance_drop'
]

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import category_encoders as ce

target_encoder = ce.TargetEncoder(
    cols=high_card_cat,
    handle_unknown='value',
    handle_missing='value'
)

onehot_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse=False  # 給 XGBoost / SMOTE 用，避免 sparse 問題
)

encoder = ColumnTransformer(
    transformers=[
        ('target_enc', target_encoder, high_card_cat),
        ('onehot', onehot_encoder, one_hot_cat),
        ('num', 'passthrough', num_cols),
    ],
    remainder='drop'   # ← 關鍵：不准偷渡任何欄位
)

X_train_encoded = encoder.fit_transform(X_train, y_train_encoded)


/Users/ninalin/venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


### 7. Distribution of dependent variable

In [ ]:
# Plot of dependent variable

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.countplot(data=student_dropout, x='Target', palette='Pastel1')
plt.title("Target Variable Count Distribution")
plt.xlabel("Target Class")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
(student_dropout['Target']
 .value_counts(normalize=True)
 .mul(100)
 .plot(kind='bar', color='steelblue'))

plt.title("Target Variable Percentage Distribution")
plt.xlabel("Target Class")
plt.ylabel("Percentage (%)")

plt.tight_layout()
plt.show()

### 7.3  Final Inspection

In [ ]:
# Final Inspection

X_train_df = pd.DataFrame(
    X_train_encoded,
    columns=encoder.get_feature_names_out()
)

print(X_train_df.head())
print(X_train_df.columns)

## 10. Train, Validation and holdout Split

to prevent dataleakage, we split from raw data and fit encoder in 

In [23]:
## Train + Validation split

X_train_modeling, X_val, y_train_modeling, y_val = train_test_split(
    X_train_encoded,         
    y_train_encoded,
    test_size=0.15,
    stratify=y_train_encoded,
    random_state=42
)


print(X_train_modeling.shape)
print(X_val.shape)
print(X_test.shape)

(3008, 193)
(531, 193)
(885, 36)


## Balanced model

## Model training + Balanced method

In [ ]:
## XGB + weight balanced

sample_weight = compute_sample_weight(
    class_weight='balanced',
    y=y_train_modeling
)

xgb_weighted = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

xgb_weighted.fit(
    X_train_modeling,
    y_train_modeling,
    sample_weight=sample_weight
)

val_pred_xgb_w = xgb_weighted.predict(X_val)

print("=== XGBoost (sample_weight balanced) ===")
print(classification_report(y_val, val_pred_xgb_w))


=== XGBoost (sample_weight balanced) ===
              precision    recall  f1-score   support

           0       0.78      0.73      0.75       171
           1       0.48      0.46      0.47        95
           2       0.79      0.83      0.81       265

    accuracy                           0.73       531
   macro avg       0.68      0.68      0.68       531
weighted avg       0.73      0.73      0.73       531



In [29]:
# XGBoost

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42
)

xgb.fit(
    X_train_modeling,
    y_train_modeling,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=30,
    verbose=True
)

val_pred_xgb = xgb.predict(X_val)

print("=== XGBoost Validation Performance ===")
print("Accuracy:", accuracy_score(y_val, val_pred_xgb))
print("\nConfusion Matrix:\n", confusion_matrix(y_val, val_pred_xgb))
print("\nClassification Report:\n", classification_report(y_val, val_pred_xgb))


[0]	validation_0-mlogloss:1.06775
[1]	validation_0-mlogloss:1.04244
[2]	validation_0-mlogloss:1.02121
[3]	validation_0-mlogloss:0.99789


/Users/ninalin/venv/lib/python3.10/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[4]	validation_0-mlogloss:0.97769
[5]	validation_0-mlogloss:0.96163
[6]	validation_0-mlogloss:0.94559
[7]	validation_0-mlogloss:0.92753
[8]	validation_0-mlogloss:0.91037
[9]	validation_0-mlogloss:0.89431
[10]	validation_0-mlogloss:0.88082
[11]	validation_0-mlogloss:0.86690
[12]	validation_0-mlogloss:0.85607
[13]	validation_0-mlogloss:0.84374
[14]	validation_0-mlogloss:0.83232
[15]	validation_0-mlogloss:0.82230
[16]	validation_0-mlogloss:0.81262
[17]	validation_0-mlogloss:0.80286
[18]	validation_0-mlogloss:0.79382
[19]	validation_0-mlogloss:0.78530
[20]	validation_0-mlogloss:0.77774
[21]	validation_0-mlogloss:0.77045
[22]	validation_0-mlogloss:0.76321
[23]	validation_0-mlogloss:0.75612
[24]	validation_0-mlogloss:0.74981
[25]	validation_0-mlogloss:0.74379
[26]	validation_0-mlogloss:0.73812
[27]	validation_0-mlogloss:0.73257
[28]	validation_0-mlogloss:0.72737
[29]	validation_0-mlogloss:0.72206
[30]	validation_0-mlogloss:0.71735
[31]	validation_0-mlogloss:0.71276
[32]	validation_0-mlogloss

In [ ]:
## Random Forest + weight balanced

rf_balanced = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'   # ← 關鍵只改這一行
)

rf_balanced.fit(X_train_modeling, y_train_modeling)

val_pred_rf_bal = rf_balanced.predict(X_val)

print("=== Random Forest (class_weight='balanced') Validation Performance ===")
print("Accuracy:", accuracy_score(y_val, val_pred_rf_bal))
print("\nConfusion Matrix:\n", confusion_matrix(y_val, val_pred_rf_bal))
print("\nClassification Report:\n", classification_report(y_val, val_pred_rf_bal))


=== Random Forest (class_weight='balanced') Validation Performance ===
Accuracy: 0.743879472693032

Confusion Matrix:
 [[128   6  37]
 [ 19  17  59]
 [ 10   5 250]]

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.75      0.78       171
           1       0.61      0.18      0.28        95
           2       0.72      0.94      0.82       265

    accuracy                           0.74       531
   macro avg       0.71      0.62      0.63       531
weighted avg       0.73      0.74      0.71       531



In [ ]:
## Gradient Boosting + undersampliung


models = {
    #"Logistic Regression": LogisticRegression(max_iter=1000),
    #"Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    #"SVM (RBF Kernel)": SVC(probability=True, random_state=42),
    #"KNN (k=5)": KNeighborsClassifier(n_neighbors=5)
}


models_under = {
    "GB + Undersampling": Pipeline([
        ('under', RandomUnderSampler(random_state=42)),
        ('model', GradientBoostingClassifier(random_state=42))
    ])
}
def evaluate_models(models_dict, X_train, y_train, X_val, y_val, results):
    for name, model in models_dict.items():
        print(f"\n===== Training {name} =====")
        
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        acc = accuracy_score(y_val, preds)
        results[name] = acc

        print(f"Accuracy: {acc:.4f}")
        print("Confusion Matrix:")
        print(confusion_matrix(y_val, preds))
        print("Classification Report:")
        print(classification_report(y_val, preds))

gb_under = models_under["GB + Undersampling"]

gb_under.fit(X_train_modeling, y_train_modeling)

results = {}
evaluate_models(models, X_train_modeling, y_train_modeling, X_val, y_val, results)



===== Training Gradient Boosting =====
Accuracy: 0.7420
Confusion Matrix:
[[127  11  33]
 [ 16  23  56]
 [ 12   9 244]]
Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.74      0.78       171
           1       0.53      0.24      0.33        95
           2       0.73      0.92      0.82       265

    accuracy                           0.74       531
   macro avg       0.70      0.64      0.64       531
weighted avg       0.73      0.74      0.72       531



## Holdout evaluation

##### We now proceed to evaluate them on the holdout set to determine the best-performing model and to ensure that our models are not overfitting.

In [ ]:
## Validation define
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score,
    log_loss
)

def validate_model(name, model, X_val, y_val, has_proba=True):
    print(f"\n========== {name} ==========")

    y_pred = model.predict(X_val)

    print("Confusion Matrix:")
    print(confusion_matrix(y_val, y_pred))

    print("Classification Report:")
    print(classification_report(y_val, y_pred))

    f1_macro = f1_score(y_val, y_pred, average='macro')
    f1_weighted = f1_score(y_val, y_pred, average='weighted')

    print(f"F1 Macro: {f1_macro:.4f}")
    print(f"F1 Weighted: {f1_weighted:.4f}")

    if has_proba:
        y_proba = model.predict_proba(X_val)
        ll = log_loss(y_val, y_proba)
        print(f"Log Loss: {ll:.4f}")
    else:
        ll = None

    return {
        "model": name,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "log_loss": ll
    }



In [ ]:
## Validation result
results = []

results.append(validate_model(
    "XGB (sample_weight balanced)",
    xgb_weighted,
    X_val, y_val,
    has_proba=True
))

results.append(validate_model(
    "XGB (early stopping)",
    xgb,
    X_val, y_val,
    has_proba=True
))

results.append(validate_model(
    "Random Forest (class_weight)",
    rf_balanced,
    X_val, y_val,
    has_proba=True
))

results.append(validate_model(
    "GB + Undersampling",
    models_under["GB + Undersampling"],
    X_val, y_val,
    has_proba=True
))



========== XGB (sample_weight balanced) ==========
Confusion Matrix:
[[125  19  27]
 [ 20  44  31]
 [ 16  28 221]]
Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.73      0.75       171
           1       0.48      0.46      0.47        95
           2       0.79      0.83      0.81       265

    accuracy                           0.73       531
   macro avg       0.68      0.68      0.68       531
weighted avg       0.73      0.73      0.73       531

F1 Macro: 0.6795
F1 Weighted: 0.7326
Log Loss: 0.6556

========== XGB (early stopping) ==========
Confusion Matrix:
[[124  13  34]
 [ 24  27  44]
 [ 10   8 247]]
Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.73      0.75       171
           1       0.56      0.28      0.38        95
           2       0.76      0.93      0.84       265

    accuracy                           0.75       531
   macro avg       0.70

In [ ]:
## Present validation result

df_results = pd.DataFrame(results)
print(df_results.sort_values(by="f1_macro", ascending=False))


                          model  f1_macro  f1_weighted  log_loss
0  XGB (sample_weight balanced)  0.679543     0.732625  0.655579
3            GB + Undersampling  0.657352     0.698550  0.730059
1          XGB (early stopping)  0.656237     0.728164  0.620367
2  Random Forest (class_weight)  0.625080     0.709192  0.679086


##### 'Proceed with XGB + sample_weight balanced, performed the best result in both training and vlaidation

## Test Data Preprocessing

In [ ]:
X_train_full = pd.concat([X_train_modeling, X_val])
y_train_full = pd.concat([y_train_modeling, y_val])
